<a href="https://colab.research.google.com/github/Pedro-Soares09/PicPay-Open-Finance-Simulator/blob/main/Emprestimo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- INSTALAÇÃO ---
!pip install pyngrok flask-cors
from pyngrok import ngrok
from flask import Flask, request, jsonify
from flask_cors import CORS
from threading import Thread
import time
from datetime import datetime

# CONFIGURAÇÃO NGROK
NGROK_TOKEN = "3DY9c1cOhOFlvLbF9xPYCzmFlf3_3cR37Xtemmpv9jrwii34N" #COLOQUE SEU TOKEN AQUI
ngrok.set_auth_token(NGROK_TOKEN)

app = Flask(__name__)
CORS(app) # CRÍTICO: Permite que navegadores e outras linguagens acessem a API

# Banco de dados na memória
banco_central_dados = {}

@app.route('/central', methods=['GET'])
def buscar():
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 📥 Consulta de dados recebida.")
    return jsonify(banco_central_dados)

@app.route('/central', methods=['POST'])
def salvar():
    global banco_central_dados
    banco_central_dados = request.json
    print(f"[{datetime.now().strftime('%H:%M:%S')}] 📤 Dados atualizados por agência externa!")
    return jsonify({"status": "sucesso", "total_registros": len(banco_central_dados)})

# Rodar servidor
def run():
    app.run(port=5000)

Thread(target=run).start()
time.sleep(2)
public_url = ngrok.connect(5000).public_url

print("\n" + "="*60)
print("🏦 BARRAMENTO OPEN FINANCE - BANCO CENTRAL ATIVO")
print("="*60)
print(f"\nLINK UNIVERSAL:")
print(f"👉 {public_url}/central")
print("\n" + "="*60)

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, timedelta
import requests

# --- CONFIGURAÇÃO DA INTERCONEXÃO ---
URL_BANCO_CENTRAL = "https://mossy-bribe-carless.ngrok-free.dev/central"

# --- INTERFACE GRÁFICA ) ---
display(HTML("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;600;700&display=swap');

    .app-container {
        background: #1a1a1a;
        font-family: 'Plus Jakarta Sans', sans-serif;
        border-radius: 30px;
        padding: 35px;
        color: #e0e0e0;
        max-width: 600px;
        margin: auto;
        box-shadow: 20px 20px 60px #0d0d0d, -20px -20px 60px #272727;
    }

    .header-box {
        background: #151515;
        border-radius: 20px;
        padding: 20px;
        text-align: center;
        margin-bottom: 30px;
        box-shadow: inset 6px 6px 12px #0a0a0a, inset -6px -6px 12px #202020;
    }

    .picpay-green { color: #11C76F; font-size: 28px; font-weight: 800; margin: 0; }
    .subtitle { color: #666; font-size: 10px; letter-spacing: 2px; text-transform: uppercase; margin-top: 5px; }

    .widget-text input, .widget-floattext input {
        background: #1a1a1a !important;
        color: white !important;
        border: none !important;
        border-radius: 12px !important;
        padding: 12px !important;
        box-shadow: inset 4px 4px 8px #0d0d0d, inset -4px -4px 8px #272727 !important;
        margin-bottom: 10px !important;
    }

    .btn-action {
        background: linear-gradient(145deg, #11C76F, #0f9d58) !important;
        color: white !important;
        font-weight: 700 !important;
        border-radius: 15px !important;
        border: none !important;
        height: 50px !important;
        margin-top: 25px !important;
        box-shadow: 6px 6px 12px #0d0d0d, -6px -6px 12px #272727 !important;
        cursor: pointer;
    }

    .card-full {
        background: #1a1a1a;
        border-radius: 20px;
        padding: 20px;
        margin: 15px 0;
        box-shadow: 8px 8px 16px #0d0d0d, -8px -8px 16px #272727;
    }
</style>
"""))

class PicPayNeumorphicV7:
    def __init__(self):
        self.out = widgets.Output()
        self.tabs = widgets.Tab(children=[self.ui_cadastro(), self.ui_extrato(), self.ui_renegociar()])
        self.tabs.set_title(0, '🏦 Novo Crédito')
        self.tabs.set_title(1, '📄 Open Finance')
        self.tabs.set_title(2, '🤝 Acordos')

    def baixar_dados(self):
        try:
            return requests.get(URL_BANCO_CENTRAL, headers={"ngrok-skip-browser-warning": "true"}, timeout=5).json()
        except:
            return {}

    def salvar_dados(self, dados):
        try:
            requests.post(URL_BANCO_CENTRAL, json=dados, timeout=5)
        except:
            with self.out:
                display(HTML("<p style='color:#f55'>⚠️ Erro de Sincronização</p>"))

    def formatar_cpf(self, change):
        nums = "".join(filter(str.isdigit, change['new']))
        fmt = f"{nums[:3]}.{nums[3:6]}.{nums[6:9]}-{nums[9:11]}" if len(nums) > 9 else nums
        change.owner.value = fmt[:14]

    def ui_cadastro(self):
        self.in_nome = widgets.Text(placeholder="Nome Completo", description="Nome:")
        self.in_mae = widgets.Text(placeholder="Nome da Mãe", description="Mãe:")
        self.in_cpf = widgets.Text(placeholder="000.000.000-00", description="CPF:")
        self.in_cpf.observe(self.formatar_cpf, names='value')
        self.in_end = widgets.Text(placeholder="Endereço Completo", description="Endereço:")
        self.in_renda = widgets.FloatText(description="Renda R$:")
        self.in_valor = widgets.FloatText(description="Solicitado:")

        btn = widgets.Button(description="EFETIVAR CONTRATAÇÃO", layout={'width': '100%'})
        btn.add_class("btn-action")
        btn.on_click(self.processar_contratacao)

        # Encapsulando HTML em widgets.HTML
        header1 = widgets.HTML("<p style='color:#666; font-size:12px; font-weight:700'>DADOS DO PROPONENTE</p>")
        header2 = widgets.HTML("<br><p style='color:#666; font-size:12px; font-weight:700'>ANÁLISE FINANCEIRA</p>")

        return widgets.VBox([header1, self.in_nome, self.in_mae, self.in_cpf, self.in_end, header2, self.in_renda, self.in_valor, btn])

    def processar_contratacao(self, b):
        with self.out:
            clear_output()
            banco = self.baixar_dados()
            cpf_limpo = "".join(filter(str.isdigit, self.in_cpf.value))

            if len(cpf_limpo) != 11:
                display(HTML("<p style='color:#f55'>❌ CPF INVÁLIDO</p>"))
                return

            venc = (datetime.now() + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
            banco[cpf_limpo] = {
                'nome': self.in_nome.value.upper(), 'mae': self.in_mae.value.upper(),
                'end': self.in_end.value.upper(), 'renda': float(self.in_renda.value),
                'valor': float(self.in_valor.value), 'status': 'ATIVO',
                'score': 850, 'vencimento': venc, 'origem': 'PicPay'
            }
            self.salvar_dados(banco)
            display(HTML("<p style='color:#11C76F; font-weight:bold'>✅ SOLICITAÇÃO ENVIADA!</p>"))

    def ui_extrato(self):
        self.out_extrato = widgets.Output()
        btn = widgets.Button(description="ATUALIZAR DADOS", layout={'width': '100%'})
        btn.add_class("btn-action")
        btn.on_click(self.renderizar_extrato)
        return widgets.VBox([btn, self.out_extrato])

    def renderizar_extrato(self, b):
        with self.out_extrato:
            clear_output()
            banco = self.baixar_dados()
            for cpf, info in banco.items():
                if not isinstance(info, dict): continue
                cor = "#11C76F" if info.get('status') == "ATIVO" else "#FF4B4B"
                display(HTML(f"""
                <div class='card-full'>
                    <b style='color:{cor}'>{info.get('status')}</b><br>
                    <b style='font-size:1.2em'>{info.get('nome')}</b><br>
                    <small>CPF: {cpf} | Score: {info.get('score')}</small><br>
                    <span style='color:{cor}; font-size:1.3em; font-weight:bold'>R$ {info.get('valor',0):.2f}</span>
                </div>
                """))

    def ui_renegociar(self):
        self.in_cpf_busca = widgets.Text(placeholder="000.000.000-00", description="CPF:")
        self.out_rec = widgets.Output()
        btn = widgets.Button(description="BUSCAR ACORDOS", layout={'width': '100%'})
        btn.add_class("btn-action")
        btn.on_click(self.buscar_divida)
        return widgets.VBox([self.in_cpf_busca, btn, self.out_rec])

    def buscar_divida(self, b):
        with self.out_rec:
            clear_output()
            banco = self.baixar_dados()
            cpf_alvo = "".join(filter(str.isdigit, self.in_cpf_busca.value))
            for cpf_b, info in banco.items():
                if "".join(filter(str.isdigit, cpf_b)) == cpf_alvo and info.get('status') == 'NEGATIVADO':
                    v_acordo = info['valor'] * 0.7
                    display(HTML(f"<div class='card-full'><h3>OFERTA: R$ {v_acordo:.2f}</h3><p>30% de desconto aplicado.</p></div>"))
                    btn_p = widgets.Button(description="PAGAR ACORDO", layout={'width': '100%'})
                    btn_p.add_class("btn-action")
                    btn_p.on_click(lambda x, c=cpf_b: self.quitar(c, banco))
                    display(btn_p)
                    return
            print("Nenhuma dívida encontrada.")

    def quitar(self, cpf, banco):
        banco[cpf].update({'status': 'ATIVO', 'valor': 0.0, 'score': 950, 'vencimento': '2099-01-01 00:00:00'})
        self.salvar_dados(banco)
        with self.out_rec: clear_output(); display(HTML("<b style='color:#11C76F'>DÍVIDA QUITADA!</b>"))

# Iniciar
instancia = PicPayNeumorphicV7()
display(HTML("<div class='app-container'><div class='header-box'><h1 class='picpay-green'>PicPayBank</h1><p class='subtitle'>SISTEMA DE GESTÃO DE CRÉDITO</p></div>"), instancia.tabs, instancia.out, HTML("</div>"))

Output()